# 02 — The Effect of Romanisation (Tables 2, 10, 11)

Quantifies what romanisation does to COMET, to tokenisation, and to the
agreement between COMET and human judgement.

| Output | Base | Content |
|---|---|---|
| **Tables 2 and 13** | 7,000 | Mean COMET per language plus TP and IP, native → romanised |
| **Tables 2, 10 and 11** | 6,995 | Spearman ρ against the human score, with Meng and Steiger tests for dependent overlapping correlations and a 5,000-round paired permutation test |
| **ANOVA** | 7,000 and 6,995 | Devanagari (HIN, MAR) vs non-Devanagari (GUJ, MAL, TAM), native and romanised |
| **HIN–GUJ** | 7,000 | Welch's *t* on the natural experiment |

**Input:** `../data/indic/indic_parity_xlmr.xlsx`
**Outputs:**
- `../results/tables/table2_comet_tp_ip.csv`
- `../results/tables/table3_correlation_tests.csv`
- `../results/tables/anova_and_hin_guj.csv`

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)

## Step 1 — Load the Workbook

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats

## Step 2 — Tables 2 and 13: Mean COMET, TP and IP

Computed on the full 7,000-segment base. Percentage changes are relative to the
native-script mean.

**The TP–IP inversion.** Romanisation drives TP *up* and IP *down* for every
language: the encoder emits more pieces, and each piece carries less
information. Tamil is the extreme case in both directions.

In [ ]:
rows = []
print("Tables 2 and 13 — mean COMET / TP / IP, native \u2192 romanised (N = 1,400)")
print(f"{'Lang':>5}  {'COMET_nat':>10}  {'COMET_rom':>10}  {'\u0394':>7}   "
      f"{'TP_nat':>7}  {'TP_rom':>7}  {'\u0394TP%':>7}   "
      f"{'IP_nat':>7}  {'IP_rom':>7}  {'\u0394IP%':>7}")
print("-" * 96)
for lang in LANG_ORDER:
    d = full[lang]
    cn, cr = d[COL_COMET_NAT].mean(), d[COL_COMET_ROM].mean()
    tn, tr = d[COL_TP_NAT].mean(), d[COL_TP_ROM].mean()
    ipn, ipr = d[COL_IP_NAT].mean(), d[COL_IP_ROM].mean()
    rows.append(dict(lang=lang, comet_nat=cn, comet_rom=cr, comet_delta=cr - cn,
                     tp_nat=tn, tp_rom=tr, tp_pct=(tr / tn - 1) * 100,
                     ip_nat=ipn, ip_rom=ipr, ip_pct=(ipr / ipn - 1) * 100))
    print(f"{lang:>5}  {cn:>10.2f}  {cr:>10.2f}  {cr - cn:>+7.2f}   "
          f"{tn:>7.3f}  {tr:>7.3f}  {(tr / tn - 1) * 100:>+7.1f}   "
          f"{ipn:>7.3f}  {ipr:>7.3f}  {(ipr / ipn - 1) * 100:>+7.1f}")

table2 = pd.DataFrame(rows).set_index("lang")

# ── Cross-verification against the paper ─────────────────────────────────────
assert abs(table2.loc["GUJ", "comet_nat"] - 85.70) < 0.01
assert abs(table2.loc["TAM", "tp_pct"] - 77.9) < 0.1
assert abs(table2.loc["TAM", "ip_pct"] - (-68.0)) < 0.1
assert (table2["tp_pct"] > 0).all(), "TP must inflate for every language"
assert (table2["ip_pct"] < 0).all(), "IP must collapse for every language"
print(f"\n\u2713 GUJ COMET_nat = {table2.loc['GUJ', 'comet_nat']:.2f} (paper: 85.70)")
print(f"\u2713 TAM \u0394TP = {table2.loc['TAM', 'tp_pct']:+.1f}% (paper: +77.9%)")
print(f"\u2713 TAM \u0394IP = {table2.loc['TAM', 'ip_pct']:+.1f}% (paper: -68.0%)")
print("\u2713 TP inflates and IP collapses for all five languages (the TP-IP inversion)")

## Step 3 — Dependent-Correlation Machinery

Native and romanised COMET are scored on the *same* segments against the *same*
human scores, so their correlations with the human score are dependent and
overlapping. Comparing them with two independent-sample tests would be wrong.
Two standard statistics apply:

- **Meng, Rosenthal & Rubin (1992)** — the conventional choice for overlapping
  dependent correlations.
- **Steiger (1980)** — an alternative estimator of the same quantity; reported
  as a robustness check, not as a second independent result.

In [ ]:
def meng_z(r12, r13, r23, n):
    """Meng, Rosenthal & Rubin (1992), two dependent overlapping correlations."""
    z12, z13 = np.arctanh(r12), np.arctanh(r13)
    rbar2 = (r12 ** 2 + r13 ** 2) / 2
    f = min((1 - r23) / (2 * (1 - rbar2)), 1.0)
    h = (1 - f * rbar2) / (1 - rbar2)
    return (z12 - z13) * np.sqrt((n - 3) / (2 * (1 - r23) * h))


def steiger_z(r12, r13, r23, n):
    """Steiger (1980), same comparison under a different covariance estimate."""
    z12, z13 = np.arctanh(r12), np.arctanh(r13)
    rm2 = (r12 ** 2 + r13 ** 2) / 2
    cov = (r23 * (1 - 2 * rm2) - 0.5 * rm2 * (1 - 2 * rm2 - r23 ** 2)) / ((1 - rm2) ** 2)
    return (z12 - z13) * np.sqrt((n - 3) / (2 - 2 * cov))


print("Dependent-correlation tests defined (Meng and Steiger).")

## Step 4 — Tables 2, 10 and 11: Correlation Collapse

Computed on the 6,995-segment working base. The permutation test draws its
5,000 sign-flips from a single generator seeded once (`SEED_PERM`), so the
counts below are reproducible but are order-dependent across languages — they
are registered as `script` rather than `verified` in `paper_numbers.yaml`.

This cell runs 25,000 rank correlations and takes roughly 40 seconds.

In [ ]:
rng = np.random.default_rng(SEED_PERM)
rows = []
print("Tables 2 and 10 — Spearman \u03c1 against the human score, native \u2192 romanised")
print(f"{'Lang':>5}  {'\u03c1_nat':>7}  {'\u03c1_rom':>7}  {'|\u0394\u03c1|':>6}  {'loss':>6}  "
      f"{'Meng z':>7}  {'Steiger z':>10}  {'perm':>10}")
print("-" * 74)
for lang in LANG_ORDER:
    d = work[lang]
    h = stats.rankdata(d["H"])
    a = stats.rankdata(d[COL_COMET_NAT])
    b = stats.rankdata(d[COL_COMET_ROM])
    r12 = np.corrcoef(a, h)[0, 1]
    r13 = np.corrcoef(b, h)[0, 1]
    r23 = np.corrcoef(a, b)[0, 1]
    n = len(d)
    obs = r12 - r13

    A = d[COL_COMET_NAT].values
    B = d[COL_COMET_ROM].values
    exceed = 0
    for _ in range(5000):
        m = rng.random(n) < 0.5
        s1 = np.corrcoef(stats.rankdata(np.where(m, B, A)), h)[0, 1]
        s2 = np.corrcoef(stats.rankdata(np.where(m, A, B)), h)[0, 1]
        if abs(s1 - s2) >= abs(obs):
            exceed += 1

    zm, zs = meng_z(r12, r13, r23, n), steiger_z(r12, r13, r23, n)
    loss = (1 - r13 / r12) * 100
    rows.append(dict(lang=lang, n=n, rho_nat=r12, rho_rom=r13, drho=abs(obs),
                     loss_pct=loss, meng_z=zm, steiger_z=zs,
                     perm_exceed=exceed, perm_rounds=5000))
    print(f"{lang:>5}  {r12:>+7.3f}  {r13:>+7.3f}  {abs(obs):>6.3f}  {loss:>5.1f}%  "
          f"{zm:>7.1f}  {zs:>10.1f}  {exceed:>5d}/5000")

table3 = pd.DataFrame(rows).set_index("lang")

# ── Cross-verification against the paper ─────────────────────────────────────
lo, hi = table3["loss_pct"].min(), table3["loss_pct"].max()
assert (table3["rho_rom"] < table3["rho_nat"]).all(), "\u03c1 must fall for every language"
assert 29 <= round(lo) and round(hi) <= 62, f"loss range {lo:.1f}-{hi:.1f}% outside 29-62%"
assert (table3["perm_exceed"] == 0).all(), "no permutation round should exceed the observed gap"
print(f"\n\u2713 Correlation loss spans {lo:.1f}% (MAL) to {hi:.1f}% (TAM) — paper: 29-62%")
print("\u2713 0/5000 permutation rounds reached the observed |\u0394\u03c1| in any language")

## Step 5 — ANOVA: How Much Variance Does Script Explain?

One-way ANOVA contrasting Devanagari (HIN, MAR) with non-Devanagari
(GUJ, MAL, TAM) COMET scores.

Reported on **both** bases. COMET is complete, so the 7,000-segment base is the
applicable one: there is no reason to discard the five segments that lack a
*human* score in a test that does not use the human score. The 6,995-restricted
value is printed alongside it for transparency.

In [ ]:
anova_rows = []
print("One-way ANOVA — Devanagari (HIN, MAR) vs non-Devanagari (GUJ, MAL, TAM)")
print(f"{'Base':>14}  {'Condition':>10}  {'N':>6}  {'F':>9}  {'p':>10}  {'\u03b7\u00b2':>7}")
print("-" * 63)
for basis, src in [("full 7,000", full), ("working 6,995", work)]:
    for cond, col in [("native", COL_COMET_NAT), ("romanised", COL_COMET_ROM)]:
        dev = pd.concat([src["HIN"], src["MAR"]])[col].dropna()
        nod = pd.concat([src["GUJ"], src["MAL"], src["TAM"]])[col].dropna()
        F, p = stats.f_oneway(dev, nod)
        allv = pd.concat([dev, nod])
        gm = allv.mean()
        ssb = len(dev) * (dev.mean() - gm) ** 2 + len(nod) * (nod.mean() - gm) ** 2
        eta2 = ssb / ((allv - gm) ** 2).sum() * 100
        anova_rows.append(dict(basis=basis, condition=cond, N=len(allv),
                               F=F, p=p, eta2_pct=eta2))
        print(f"{basis:>14}  {cond:>10}  {len(allv):>6}  {F:>9.1f}  {p:>10.2e}  {eta2:>6.1f}%")

anova = pd.DataFrame(anova_rows)

f_7000 = anova.query("basis == 'full 7,000' and condition == 'native'")["F"].iloc[0]
f_6995 = anova.query("basis == 'working 6,995' and condition == 'native'")["F"].iloc[0]
e_nat = anova.query("basis == 'full 7,000' and condition == 'native'")["eta2_pct"].iloc[0]
e_rom = anova.query("basis == 'full 7,000' and condition == 'romanised'")["eta2_pct"].iloc[0]
reduction = (e_nat - e_rom) / e_nat * 100

# ── Cross-verification against the paper ─────────────────────────────────────
assert abs(f_7000 - 2080.8) < 0.1, f"native F (7,000) = {f_7000:.1f}, expected 2080.8"
assert abs(e_nat - 22.9) < 0.1, f"native \u03b7\u00b2 = {e_nat:.1f}%, expected 22.9%"
assert abs(reduction - 93.3) < 0.2, f"variance reduction = {reduction:.1f}%, expected 93.3%"
print(f"\n\u2713 Native F (7,000 base) = {f_7000:.1f} (paper: 2,080.8)")
print(f"\u2713 Native \u03b7\u00b2 = {e_nat:.1f}% (paper: 22.9%)")
print(f"\u2713 Romanisation removes {reduction:.1f}% of between-script variance "
      f"(\u03b7\u00b2 {e_nat:.1f}% \u2192 {e_rom:.1f}%) (paper: 93.3%)")
print(f"\n  Reported on both bases for transparency: F = {f_7000:.1f} on 7,000 "
      f"and F = {f_6995:.1f} on 6,995.")
print( "  The ANOVA uses metric-side quantities only, so the 7,000-segment base "
       "applies.")

## Step 6 — The HIN–GUJ Natural Experiment

Hindi and Gujarati are both Indo-Aryan and both written in Brahmic scripts, but
COMET separates them by roughly 14 points in native script. If that gap were
linguistic it should survive romanisation. It does not: romanising both sides
collapses it to about 2 points.

Welch's *t* (unequal variances) on all 1,400 segments per language.

In [ ]:
hin_guj_rows = []
print("HIN\u2013GUJ natural experiment — Welch's t, N = 1,400 per language")
print(f"{'Condition':>10}  {'GUJ mean':>9}  {'HIN mean':>9}  {'gap':>7}  {'t':>8}  {'p':>11}")
print("-" * 60)
for cond, col in [("native", COL_COMET_NAT), ("romanised", COL_COMET_ROM)]:
    g, h_ = full["GUJ"][col], full["HIN"][col]
    t, p = stats.ttest_ind(g, h_, equal_var=False)
    hin_guj_rows.append(dict(condition=cond, guj_mean=g.mean(), hin_mean=h_.mean(),
                             gap=g.mean() - h_.mean(), welch_t=t, p=p))
    print(f"{cond:>10}  {g.mean():>9.2f}  {h_.mean():>9.2f}  "
          f"{g.mean() - h_.mean():>7.2f}  {t:>8.2f}  {p:>11.2e}")

hin_guj = pd.DataFrame(hin_guj_rows).set_index("condition")
gap_nat = hin_guj.loc["native", "gap"]
gap_rom = hin_guj.loc["romanised", "gap"]
t_nat = hin_guj.loc["native", "welch_t"]
t_rom = hin_guj.loc["romanised", "welch_t"]
collapse = (gap_nat - gap_rom) / gap_nat * 100

# ── Cross-verification against the paper ─────────────────────────────────────
assert abs(t_nat - 27.4) < 0.05, f"native Welch t = {t_nat:.2f}, expected 27.4"
assert abs(t_rom - 6.97) < 0.05, f"romanised Welch t = {t_rom:.2f}, expected 6.97"
assert abs(gap_nat - 13.95) < 0.01, f"native gap = {gap_nat:.2f}, expected 13.95"
assert abs(gap_rom - 2.29) < 0.01, f"romanised gap = {gap_rom:.2f}, expected 2.29"
print(f"\n\u2713 Native Welch t = {t_nat:.2f} (reported: 27.4)")
print(f"\u2713 Romanised Welch t = {t_rom:.2f} (reported: 6.97)")
print(f"\u2713 Gap collapses {gap_nat:.2f} \u2192 {gap_rom:.2f} pts "
      f"({collapse:.0f}% reduction) (paper: 84%)")

## Step 7 — Save

In [ ]:
paths = []
for name, frame in [("table2_comet_tp_ip", table2),
                    ("table10_correlation_tests", table3)]:
    p = TABLES_DIR / f"{name}.csv"
    frame.to_csv(p)
    paths.append(p)

combined = pd.concat([
    anova.assign(analysis="anova"),
    hin_guj.reset_index().assign(analysis="hin_guj"),
], ignore_index=True)
p = TABLES_DIR / "anova_and_hin_guj.csv"
combined.to_csv(p, index=False)
paths.append(p)

for p in paths:
    print(f"Saved \u2192 {p}")

## Step 9 — Output Manifest

In [ ]:
print("=== Notebook 02 — output manifest ===")
for name in ["table2_comet_tp_ip.csv", "table10_correlation_tests.csv",
             "anova_and_hin_guj.csv"]:
    print(f"  {name}")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1